# Lesson 4: Persistence and Streaming

In [ ]:
from dotenv import load_dotenv
from utils import printer
_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [ ]:
ollama_service = "http://localhost:11434/v1/"
model_name = "qwen3-vl:4b-instruct"

In [ ]:
tool = TavilySearch(max_results=2)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

memory = InMemorySaver()

In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(MessagesState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_edge(START, "llm")
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")

        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: MessagesState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: MessagesState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: MessagesState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print("Calling: ", end="")
            printer(t)
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
SYSTEM_MESSAGE = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(base_url=ollama_service, model=model_name)
agent = Agent(model, [tool], system=SYSTEM_MESSAGE, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [ ]:
thread: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
for event in agent.graph.stream({"messages": messages}, thread):
    for v in event.values():
        printer(v['messages'])

In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread: RunnableConfig = {"configurable": {"thread_id": "1"}}

for event in agent.graph.stream({"messages": messages}, thread):
    for v in event.values():
        printer(v)

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread: RunnableConfig = {"configurable": {"thread_id": "1"}}

for event in agent.graph.stream({"messages": messages}, thread):
    for v in event.values():
        printer(v)

In [ ]:
messages = [HumanMessage(content="Which one is warmer?")]
thread: RunnableConfig = {"configurable": {"thread_id": "2"}}

for event in agent.graph.stream({"messages": messages}, thread):
    for v in event.values():
        printer(v)

## Streaming tokens

In [ ]:
memory = InMemorySaver()
agent = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread: RunnableConfig = {"configurable": {"thread_id": "4"}}

async for event in agent.graph.astream_events({"messages": messages}, thread):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")